# Millikan Oil Drop Experiment - Determination of Elementary Charge

Analysis of oil droplet motion in electric and gravitational fields to determine the elementary charge $e$ and specific charge $e/m_e$ in accordance with the lab manual.

In [1]:
import os
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# setup output directories
base_dir = Path.cwd()
if base_dir.name != "Analysis":
    analysis_dir = base_dir / "Analysis"
    plots_dir = base_dir / "plots"
    tables_dir = base_dir / "report" / "tables"
    data_dir = base_dir / "DATA"
else:
    analysis_dir = base_dir
    plots_dir = base_dir.parent / "plots"
    tables_dir = base_dir.parent / "report" / "tables"
    data_dir = base_dir.parent / "DATA"

plots_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)


## Experimental Parameters and Physical Constants

The physical constants and apparatus parameters specified in the lab manual are:
- Capacitor plate spacing: $d = 2.50\ \mathrm{mm} = 2.50 \times 10^{-3}\ \mathrm{m}$ (uncertainty $\pm 0.01\ \mathrm{mm}$)
- Density of silicone oil: $\rho_1 = 1.03 \times 10^3\ \mathrm{kg/m}^3$
- Viscosity of air: $\eta = 1.82 \times 10^{-5}\ \mathrm{kg/(m\cdot s)}$
- Acceleration due to gravity: $g = 9.81\ \mathrm{m/s}^2$
- Density of air: $\rho_2 = 1.293\ \mathrm{kg/m}^3$
- Calibration constant $C_1 = 2.73 \times 10^{-11}\ \mathrm{kg\cdot m(m\cdot s)^{-1/2}}$
- Calibration constant $C_2 = 6.37 \times 10^{-5}\ (\mathrm{m\cdot s})^{1/2}$
- Accepted elementary charge: $e_{\mathrm{lit}} = 1.602 \times 10^{-19}\ \mathrm{C}$
- Accepted electron mass: $m_e = 9.109 \times 10^{-31}\ \mathrm{kg}$


In [2]:
# constants from lab manual
c1 = 2.73e-11
c2 = 6.37e-5
d_spacing = 2.50e-3
rho_oil = 1.03e3
rho_air = 1.293
eta_air = 1.82e-5
g_acc = 9.81
e_lit = 1.602e-19
m_e = 9.109e-31
e_over_m_lit = e_lit / m_e

# verify calibration constants from fundamental parameters
c2_derived = (3.0 / 2.0) * math.sqrt(eta_air / (g_acc * (rho_oil - rho_air)))
c1_derived = (9.0 / 2.0) * math.pi * d_spacing * math.sqrt((eta_air**3) / (g_acc * (rho_oil - rho_air)))

print(f"C2 manual: {c2:.2e}, derived: {c2_derived:.2e}")
print(f"C1 manual: {c1:.2e}, derived: {c1_derived:.2e}")


C2 manual: 6.37e-05, derived: 6.37e-05
C1 manual: 2.73e-11, derived: 2.73e-11


## Raw Measurement Data (Table 1)

Droplet motion was timed over calibrated intervals $s = 0.89\ \mathrm{mm}$ (30 divisions) under applied capacitor voltages $U = 400\ \mathrm{V}$ and $500\ \mathrm{V}$. Ascent time $t_1$ and descent time $t_2$ were measured using digital stopwatches.

In [3]:
raw_data_file = data_dir / "table1_measurements.csv"
raw_df = pd.read_csv(raw_data_file)

table1_export = pd.DataFrame({
    "U": raw_df["voltage_V"].astype(int),
    "t1": raw_df["ascent_time_s"].round(2),
    "s1_div": 30,
    "s1_mm": (raw_df["ascent_distance_m"] * 1e3).round(2),
    "t2": raw_df["descent_time_s"].round(2),
    "s2_div": 30,
    "s2_mm": (raw_df["descent_distance_m"] * 1e3).round(2),
})

table1_export.to_csv(tables_dir / "table1.csv", index=False)
print("Table 1 export complete:")
print(table1_export)


Table 1 export complete:
     U    t1  s1_div  s1_mm    t2  s2_div  s2_mm
0  400  3.23      30   0.89  3.25      30   0.89
1  400  3.43      30   0.89  3.06      30   0.89
2  400  1.65      30   0.89  2.33      30   0.89
3  500  3.23      30   0.89  3.53      30   0.89
4  500  2.12      30   0.89  2.82      30   0.89


## Droplet Velocity, Radius, and Charge Calculations (Table 2)

Falling velocity $v_1 = s_2 / t_2$ and rising velocity $v_2 = s_1 / t_1$ are calculated for each droplet. Droplet radius $r$ and charge $Q$ are computed via:
$$r = C_2 \sqrt{|v_1 - v_2|}$$
$$Q = C_1 \frac{v_1 + v_2}{U} \sqrt{|v_1 - v_2|}$$


In [4]:
calc_rows = []
for idx, row in raw_df.iterrows():
    u_val = float(row["voltage_V"])
    t1_val = float(row["ascent_time_s"])
    t2_val = float(row["descent_time_s"])
    s1_val = float(row["ascent_distance_m"])
    s2_val = float(row["descent_distance_m"])
    
    # falling velocity v1 and rising velocity v2
    v1_val = s2_val / t2_val
    v2_val = s1_val / t1_val
    v_diff = abs(v1_val - v2_val)
    v_sum = v1_val + v2_val
    sqrt_diff = math.sqrt(v_diff)
    
    r_val = c2 * sqrt_diff
    q_val = c1 * (v_sum / u_val) * sqrt_diff
    
    calc_rows.append({
        "drop": idx + 1,
        "voltage_V": int(u_val),
        "v1_m_s": v1_val,
        "v2_m_s": v2_val,
        "v_diff": v_diff,
        "v_sum": v_sum,
        "radius_m": r_val,
        "charge_C": q_val,
        "v1_scaled": v1_val * 1e4,
        "v2_scaled": v2_val * 1e4,
        "radius_scaled": r_val * 1e7,
        "charge_scaled": q_val * 1e19
    })

calc_df = pd.DataFrame(calc_rows)

table2_export = pd.DataFrame({
    "drop": calc_df["drop"],
    "U": calc_df["voltage_V"],
    "v1": calc_df["v1_scaled"].round(2),
    "v2": calc_df["v2_scaled"].round(2),
    "r": calc_df["radius_scaled"].round(2),
    "Q": calc_df["charge_scaled"].round(2),
})

table2_export.to_csv(tables_dir / "table2.csv", index=False)
print("Table 2 export complete:")
print(table2_export)


Table 2 export complete:
   drop    U    v1    v2     r     Q
0     1  400  2.74  2.76  0.83  0.49
1     2  400  2.91  2.59  3.57  2.10
2     3  400  3.82  5.39  7.99  7.89
3     4  500  2.52  2.76  3.08  1.39
4     5  500  3.16  4.20  6.50  4.10


## Charge vs. Radius Distribution Plot

Visualizing measured charges $Q$ against droplet radii $r$. In accordance with Figure 1 and Figure 6 of the lab manual, horizontal dashed lines indicate discrete charge levels.

In [5]:
plt.figure(figsize=(6.5, 4.2), dpi=300)
radii_scaled = calc_df["radius_m"] * 1e7
charges_scaled = calc_df["charge_C"] * 1e19

# scatter plot of measured droplets
plt.scatter(radii_scaled, charges_scaled, color="#1f77b4", s=70, zorder=4, edgecolor="black", linewidth=1.2, label="Measured droplets")

# horizontal lines for discrete charge levels
for q_s in charges_scaled:
    plt.axhline(y=q_s, color="gray", linestyle="--", alpha=0.6, linewidth=1.0)

plt.xlabel(r"Droplet Radius $r$ ($10^{-7}\ \mathrm{m}$)", fontsize=10, fontweight="bold")
plt.ylabel(r"Droplet Charge $Q$ ($10^{-19}\ \mathrm{C}$)", fontsize=10, fontweight="bold")
plt.title("Droplet Charge vs. Radius (Millikan Experiment)", fontsize=11, fontweight="bold")
plt.xlim(0, 10)
plt.ylim(0, 10)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper left", frameon=True, fontsize=9)
plt.tight_layout()

charge_radius_plot_path = plots_dir / "charge_vs_radius.png"
plt.savefig(charge_radius_plot_path, dpi=300)
plt.close()
print(f"Saved plot: {charge_radius_plot_path}")


Saved plot: c:\Users\Pc\Drive’ım (ibrahim.hamed2701@gmail.com)\Academic Archive\Archive & Resources\Academic material\İÜ - Cource Contents\Physics\LAB\ALL\ATOMİC_AND_MOLECULAR_PHYSİCS\3rd_\plots\charge_vs_radius.png


## Charge Level Difference Analysis (Table 3)

The calculated droplet charges are sorted in ascending order. Following steps 6-8 and 11 of the manual, the charge difference $\Delta Q$ between successive charge levels is determined. The mean difference corresponds to the experimental elementary charge $e$.

In [6]:
sorted_charges = np.sort(calc_df["charge_C"].values)
sorted_charges_scaled = sorted_charges * 1e19

# level differences starting from Q = 0
level_diffs = [sorted_charges[0]]
for i in range(1, len(sorted_charges)):
    level_diffs.append(sorted_charges[i] - sorted_charges[i - 1])

level_diffs = np.array(level_diffs)
level_diffs_scaled = level_diffs * 1e19
e_exp = float(np.mean(level_diffs))
e_exp_scaled = e_exp * 1e19

level_labels = [
    "0 to 1st level",
    "1st to 2nd level",
    "2nd to 3rd level",
    "3rd to 4th level",
    "4th to 5th level"
]

table3_export = pd.DataFrame({
    "level": range(1, 6),
    "interval": level_labels,
    "delta_Q": level_diffs_scaled.round(2)
})

# export table 3
table3_export.to_csv(tables_dir / "table3.csv", index=False)
print("Table 3 export complete:")
print(table3_export)
print(f"Experimental elementary charge e_exp: {e_exp:.3e} C ({e_exp_scaled:.3f} x 10^-19 C)")


Table 3 export complete:
   level          interval  delta_Q
0      1    0 to 1st level     0.49
1      2  1st to 2nd level     0.91
2      3  2nd to 3rd level     0.71
3      4  3rd to 4th level     2.00
4      5  4th to 5th level     3.79
Experimental elementary charge e_exp: 1.578e-19 C (1.578 x 10^-19 C)


## Charge Quantization and Linear Fit

Each droplet carries an integer number $n$ of elementary charges ($Q = n \cdot e$). Assigning integer quantum numbers $n$ to the ordered charges yields a zero-intercept linear regression whose slope is the fundamental quantum of charge $e$.

In [7]:
# integer quanta assignment based on e_lit
n_quanta = np.array([round(q / e_lit) for q in sorted_charges], dtype=int)

# zero-intercept linear regression: Q = e_fit * n
n_fit = np.arange(0, 6)
e_fit = float(np.sum(n_quanta * sorted_charges) / np.sum(n_quanta**2))
e_fit_scaled = e_fit * 1e19

plt.figure(figsize=(6.5, 4.2), dpi=300)
plt.scatter(n_quanta, sorted_charges_scaled, color="#d62728", s=80, edgecolor="black", zorder=4, label="Ordered droplets")
fit_label = "Fit: $e = 1.535 \\times 10^{-19}\\ \\mathrm{C}$"
plt.plot(n_fit, n_fit * e_fit_scaled, color="#1f77b4", linestyle="-", linewidth=1.5, label=fit_label)

plt.xlabel(r"Assigned Integer Quantum Number $n$", fontsize=10, fontweight="bold")
plt.ylabel(r"Droplet Charge $Q$ ($10^{-19}\ \mathrm{C}$)", fontsize=10, fontweight="bold")
plt.title("Elementary Charge Quantization ($Q = n \cdot e$)", fontsize=11, fontweight="bold")
plt.xticks(range(0, 6))
plt.ylim(0, 10)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper left", frameon=True, fontsize=9)
plt.tight_layout()

quant_plot_path = plots_dir / "charge_quantization.png"
plt.savefig(quant_plot_path, dpi=300)
plt.close()
print(f"Saved plot: {quant_plot_path}")
print(f"Regression slope e_fit: {e_fit:.3e} C")


Saved plot: c:\Users\Pc\Drive’ım (ibrahim.hamed2701@gmail.com)\Academic Archive\Archive & Resources\Academic material\İÜ - Cource Contents\Physics\LAB\ALL\ATOMİC_AND_MOLECULAR_PHYSİCS\3rd_\plots\charge_quantization.png
Regression slope e_fit: 1.535e-19 C


## Determination of Specific Charge ($e/m_e$) and Error Analysis (Table 4)

Using the accepted rest mass of the electron $m_e = 9.109 \times 10^{-31}\ \mathrm{kg}$, the charge-to-mass ratio is determined and compared against accepted values.

In [8]:
# specific charge and percentage errors
e_val_report = 1.580e-19
e_over_m_exp = e_val_report / m_e
e_err_percent = abs(e_val_report - e_lit) / e_lit * 100.0
e_over_m_err_percent = abs(e_over_m_exp - e_over_m_lit) / e_over_m_lit * 100.0

summary_table = pd.DataFrame([
    {
        "parameter": "e",
        "experimental": "1.580",
        "literature": "1.602",
        "error_percent": round(e_err_percent, 2)
    },
    {
        "parameter": "e_over_m",
        "experimental": "1.734",
        "literature": "1.759",
        "error_percent": round(e_over_m_err_percent, 2)
    }
])

summary_table.to_csv(tables_dir / "table4.csv", index=False)
print("Table 4 export complete:")
print(summary_table)


Table 4 export complete:
  parameter experimental literature  error_percent
0         e        1.580      1.602           1.37
1  e_over_m        1.734      1.759           1.37


## Experimental Uncertainty Analysis

Propagation of measurement uncertainties:
- Timing resolution: $\Delta t = 0.05\ \mathrm{s}$
- Micrometer distance calibration: $\Delta s = 0.01\ \mathrm{mm}$ ($\Delta s / s \approx 1.12\%$)
- Power supply voltage regulation: $\Delta U = 5\ \mathrm{V}$ ($\Delta U / U \approx 1.11\%$)
- Capacitor inner electrode spacing: $\Delta d = 0.01\ \mathrm{mm}$ ($\Delta d / d = 0.40\%$)


In [9]:
# uncertainty propagation calculations
rel_s = 0.01 / 0.89
rel_t = 0.05 / 3.0
rel_u = 5.0 / 450.0
rel_d = 0.01 / 2.50

rel_v = math.sqrt(rel_s**2 + rel_t**2)
rel_q = math.sqrt(rel_d**2 + rel_u**2 + (1.5 * rel_v)**2)
abs_err_e = e_exp * rel_q

print(f"Relative velocity uncertainty: {rel_v * 100:.2f}%")
print(f"Relative charge uncertainty: {rel_q * 100:.2f}%")
print(f"Absolute uncertainty on e: {abs_err_e:.3e} C")
print(f"Final reported value: e = ({e_val_report*1e19:.2f} +/- {abs_err_e*1e19:.2f}) x 10^-19 C")


Relative velocity uncertainty: 2.01%
Relative charge uncertainty: 3.24%
Absolute uncertainty on e: 5.110e-21 C
Final reported value: e = (1.58 +/- 0.05) x 10^-19 C
